# LangChain ConversationSummaryMemory 

**Goal:** Understand how LangChain summarizes an ongoing conversation, how the summary is updated, and how an LLM uses that summary in later turns.

This notebook follows the uploaded `Langchain_Summary_memory.ipynb`, which introduces:
- `ConversationSummaryMemory`
- Progressive summarization
- `ChatGroq`
- `ConversationChain`
- Multiple turns of conversation

The explanations and practice examples below expand those same concepts for classroom learning.

> **Security:** The uploaded notebook contained a hardcoded Groq API key. Do not reuse or share that key. Revoke/rotate it in your Groq account and use an environment variable for the replacement.

## 1. The problem: an LLM does not automatically remember earlier turns

Imagine this conversation:

- Human: My name is Omkar. I am an AI Engineer.
- AI: Nice to meet you, Omkar.
- Human: What is my profession?

If the model receives only the last question, it may not know the profession. A chat application must provide relevant previous context along with the current question.

LangChain memory helps manage that context.

## 2. What is ConversationSummaryMemory?

**ConversationSummaryMemory** is a memory mechanism that uses an LLM to create and progressively update a concise summary of the conversation.

It does not need to keep sending every old message as-is. Instead, it maintains a summary that captures useful details.

### Meaning of the name

| Word | Meaning |
|---|---|
| Conversation | The discussion between the human and AI |
| Summary | A shorter version containing important information |
| Memory | Information retained for later turns |

**Simple analogy:** Instead of keeping every page of a student's discussion, a teacher writes a short note containing the important points and updates that note after each new discussion.

## 3. Understand progressive summarization

The uploaded notebook describes this process:

1. Start with the current summary.
2. Take the new lines of conversation.
3. Ask the LLM to combine the old summary and new lines.
4. Produce an updated summary.

### Example

**Current summary:**
> The human is interested in artificial intelligence.

**New conversation:**
- Human: Why is AI useful?
- AI: AI can help people automate repetitive tasks.

**New summary:**
> The human is interested in artificial intelligence and has learned that AI can automate repetitive tasks.

The summary grows in meaning, not necessarily by copying every sentence.

## 4. Visualize the execution flow

```text
Human sends message
        |
        v
ConversationChain receives input
        |
        v
ConversationSummaryMemory provides current summary
        |
        v
LLM receives summary + current question
        |
        v
LLM generates an answer
        |
        v
New human/AI exchange is summarized
        |
        v
Updated summary is retained for the next turn
```

**Important distinction:** The LLM generates the answer and also helps update the summary. The memory object manages the summary and supplies it to the chain.

## 5. Install the required packages

Run this cell in Jupyter. If the packages are already installed, pip may report that they are satisfied.

In [ ]:
%pip install -U langchain-classic langchain-groq

## 6. Configure the Groq API key safely

Set the key in your operating system or notebook environment. Do not write the real key directly in the notebook.

For a temporary Jupyter session, you can set an environment variable before running the model cell:

```python
import os
os.environ["GROQ_API_KEY"] = "paste-your-new-key-here"
```

Avoid saving a real key in a notebook that you share or push to GitHub.

If the key is missing, the next cell will stop with a clear error.

In [ ]:
import os

if not os.getenv("GROQ_API_KEY","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"):
    raise ValueError(
        "GROQ_API_KEY is not set. Set it as an environment variable first."
    )

print("Groq API key is available.")

## 7. Import the required classes

| Import | Purpose |
|---|---|
| `ConversationChain` | Connects the model and memory for a conversation |
| `ConversationSummaryMemory` | Maintains a progressive conversation summary |
| `ChatGroq` | Calls a chat model hosted by Groq |

In [3]:
import os
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationSummaryMemory
from langchain_groq import ChatGroq

## 8. Initialize the chat model

The original notebook uses `qwen/qwen3.8-27b`. Model availability can change, so the model name is configurable below. Set `GROQ_MODEL` to a model currently available to your Groq account.

`temperature=0.5` controls response variability. Lower values generally make responses more consistent.

In [4]:
llm_model = ChatGroq(
    model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
    api_key=os.getenv("GROQ_API_KEY", "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"),
    temperature=0.5
)

print("Chat model initialized.")

Chat model initialized.


## 9. Create the summary memory object

```python
summary_memory = ConversationSummaryMemory(llm=llm_model)
```

Explanation:

- `summary_memory` is the Python variable.
- `ConversationSummaryMemory(...)` creates the memory object.
- `llm=llm_model` supplies the LLM that will write/update the summary.

The LLM is needed because summarization is a language task: it must understand the messages and produce a shorter version.

In [5]:
summary_memory = ConversationSummaryMemory(llm=llm_model)

print(type(summary_memory).__name__)

ConversationSummaryMemory


C:\Users\vvard\AppData\Local\Temp\ipykernel_12396\1955011931.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  summary_memory = ConversationSummaryMemory(llm=llm_model)


## 10. Connect memory and LLM using ConversationChain

This follows the structure of your uploaded notebook.

- `llm=llm_model`: which model should generate answers?
- `memory=summary_memory`: which memory should manage conversation context?
- `verbose=True`: show additional chain execution details.

**Note:** `ConversationChain` and `ConversationSummaryMemory` are legacy-style APIs. They are useful here because they are the APIs used in your source notebook. For new production projects, check the current LangChain documentation for supported message-history patterns.

In [6]:
conversation = ConversationChain(
    llm=llm_model,
    memory=summary_memory,
    verbose=True
)

C:\Users\vvard\AppData\Local\Temp\ipykernel_12396\173182482.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(


## 11. First conversation: introduce yourself

The `predict()` method sends an input to the chain and returns the model's response.

Run this cell:

In [8]:
response1 = conversation.predict(
    input="Hi, my name is veera. I am an fde Engineer."
)

print(response1)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Omar introduces himself as an AI Engineer. The AI greets him, describes the field as a fascinating intersection of theory, engineering, and data artistry, and asks about his specific focus, inquiring whether he works on research or application sides and what he is currently building.
Human: Hi, my name is veera. I am an fde Engineer.
AI:

> Finished chain.
Hello Veera! It is wonderful to meet you.

I have to admit, when I read "fde Engineer," my internal pattern-matching circuits spun their wheels for a split second trying to pin down the exact acronym, as it isn't one of the most universally standardized titles in the general engineering lexicon like "SWE" or "ML Engi

### What should you understand after the first turn?

The chain has received the introduction and generated an answer. The memory then uses the LLM to update its summary based on the conversation.

The exact summary wording depends on the model, so your output may differ from the example.

## 12. Second conversation: add new information

This mirrors the second message in your uploaded notebook: `I'm teaching LangChain`.

Run:

In [9]:
response2 = conversation.predict(
    input="I am teaching devops also."
)

print(response2)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Omar introduces himself as an AI Engineer. The AI greets him, describes the field as a fascinating intersection of theory, engineering, and data artistry, and asks about his specific focus, inquiring whether he works on research or application sides and what he is currently building. Veera then introduces herself as an FDE Engineer. The AI greets her, expresses uncertainty about the specific meaning of the acronym "FDE" (suggesting possibilities like Full-Stack Developer, Frontend Development, or Field/Functional Design Engineer), and asks for clarification. Additionally, the AI inquires about Veera’s daily workflow, specifically whether she works with legacy code or b

The updated summary should now capture both relevant facts:
- The human is an AI Engineer.
- The human is teaching LangChain.

It may also include other useful context from the conversation.

## 13. Ask a question that depends on earlier messages

Now ask about information from the earlier turns.

In [10]:
response3 = conversation.predict(
    input="What is my profession, and what am I teaching?"
)

print(response3)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Omar introduces himself as an AI Engineer. The AI greets him, describes the field as a fascinating intersection of theory, engineering, and data artistry, and asks about his specific focus, inquiring whether he works on research or application sides and what he is currently building. Veera then introduces herself as an FDE Engineer. The AI greets her, expresses uncertainty about the specific meaning of the acronym "FDE" (suggesting possibilities like Full-Stack Developer, Frontend Development, or Field/Functional Design Engineer), and asks for clarification. Additionally, the AI inquires about Veera’s daily workflow, specifically whether she works with legacy code or b

**Expected meaning of the answer:** The person is an fde Engineer and is teaching devops also.

The exact sentence is generated by the LLM and may vary.

## 14. Inspect the summary stored in memory

`load_memory_variables({})` retrieves the memory variables in the format expected by the chain.

The empty dictionary `{}` means we are not supplying any additional input values for this inspection.

In [11]:
memory_data = summary_memory.load_memory_variables({})

print(memory_data)

{'history': 'Omar introduces himself as an AI Engineer. The AI greets him, describes the field as a fascinating intersection of theory, engineering, and data artistry, and asks about his specific focus, inquiring whether he works on research or application sides and what he is currently building. Veera then introduces herself as an FDE Engineer. The AI greets her, expresses uncertainty about the specific meaning of the acronym "FDE" (suggesting possibilities like Full-Stack Developer, Frontend Development, or Field/Functional Design Engineer), and asks for clarification. Additionally, the AI inquires about Veera’s daily workflow, specifically whether she works with legacy code or builds new systems, and asks about her current tech stack, noting the integration of AI tools into development. Veera adds that she also teaches DevOps. The AI praises this addition, highlighting the dynamic nature of the field at the intersection of software development and IT operations, particularly with th

### Print only the summary text

The default memory key is commonly `history` for this legacy API. The following cell checks the returned dictionary before displaying it.

In [12]:
memory_data = summary_memory.load_memory_variables({})

print(memory_data.get("history", "No history value was returned."))

Omar introduces himself as an AI Engineer. The AI greets him, describes the field as a fascinating intersection of theory, engineering, and data artistry, and asks about his specific focus, inquiring whether he works on research or application sides and what he is currently building. Veera then introduces herself as an FDE Engineer. The AI greets her, expresses uncertainty about the specific meaning of the acronym "FDE" (suggesting possibilities like Full-Stack Developer, Frontend Development, or Field/Functional Design Engineer), and asks for clarification. Additionally, the AI inquires about Veera’s daily workflow, specifically whether she works with legacy code or builds new systems, and asks about her current tech stack, noting the integration of AI tools into development. Veera adds that she also teaches DevOps. The AI praises this addition, highlighting the dynamic nature of the field at the intersection of software development and IT operations, particularly with the rise of Inf

## 15. Add more information and observe the summary update

Let's continue the same conversation. Each call uses the same `conversation` object, so it uses the same memory object.

In [13]:
conversation.predict(input="I work with Python and cloud technologies.")
conversation.predict(input="I also teach students through practical examples.")

print(summary_memory.load_memory_variables({}).get("history"))



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Omar introduces himself as an AI Engineer. The AI greets him, describes the field as a fascinating intersection of theory, engineering, and data artistry, and asks about his specific focus, inquiring whether he works on research or application sides and what he is currently building. Veera then introduces herself as an FDE Engineer. The AI greets her, expresses uncertainty about the specific meaning of the acronym "FDE" (suggesting possibilities like Full-Stack Developer, Frontend Development, or Field/Functional Design Engineer), and asks for clarification. Additionally, the AI inquires about Veera’s daily workflow, specifically whether she works with legacy code or b

Ask the model to recall several details:

In [14]:
response4 = conversation.predict(
    input="Summarize the important details I have shared about my work."
)

print(response4)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Omar introduces himself as an AI Engineer. The AI greets him, describes the field as a fascinating intersection of theory, engineering, and data artistry, and asks about his specific focus, inquiring whether he works on research or application sides and what he is currently building. Veera then introduces herself as an FDE Engineer. The AI greets her, expresses uncertainty about the specific meaning of the acronym "FDE" (suggesting possibilities like Full-Stack Developer, Frontend Development, or Field/Functional Design Engineer), and asks for clarification. Additionally, the AI inquires about Veera’s daily workflow, specifically whether she works with legacy code or b

## 16. What does `ConversationSummaryMemory` save?

Conceptually, it maintains a **summary of the conversation**, rather than requiring the full raw conversation to be included in the prompt every time.

A summary may preserve:
- Names and roles introduced in the chat
- Important preferences or goals
- Topics discussed
- Decisions or useful facts

A summary is not guaranteed to preserve every small detail. If a detail is omitted from the summary, the model may not be able to recall it later.

## 17. Compare Buffer Memory and Summary Memory

| Feature | ConversationBufferMemory | ConversationSummaryMemory |
|---|---|---|
| Stores | Full conversation history | A progressively updated summary |
| Main idea | Keep the messages | Condense the messages |
| Context size | Usually grows with every turn | Summary is generally shorter |
| Detail preservation | Keeps original messages in memory | May omit minor details |
| LLM needed to update memory | No summarization LLM required | Yes, LLM creates/updates summary |

**Classroom analogy:**
- Buffer Memory = keep the complete classroom conversation transcript.
- Summary Memory = keep a teacher's concise notes about the important points.

## 18. Understand the prompt used for summarization

The source notebook includes the following summarization pattern:

```text
Current summary: {summary}

New lines of conversation:
{new_lines}

New summary:
```

Meaning:
- `{summary}` is the summary already created.
- `{new_lines}` are the latest conversation messages.
- The LLM combines them and writes the updated summary.

This is the core idea behind progressive summarization.

## 19. Important: summary memory is not permanent database storage

A memory object in a running Python process is generally in-process state. If the notebook kernel or application restarts, that in-memory state is lost unless you save it elsewhere.

For a real application:
- Use a database or persistent chat-history store.
- Associate each conversation with a session/user identifier.
- Apply access controls so one user's history is not exposed to another.
- Consider retention and privacy requirements.

A summary is also not a perfect record. It can compress or omit information.

## 20. Common beginner mistakes

| Mistake | Why it is a problem | What to do |
|---|---|---|
| Creating a new memory object for every message | Previous context is not reused | Reuse the same memory/chain for that conversation |
| Using a different chain each time without reusing memory | The new chain may have no prior summary | Reuse the same chain or persistent store |
| Hardcoding an API key | The key can leak through notebook sharing/Git | Use environment variables |
| Assuming every detail is remembered | Summaries can omit details | Ask for important facts explicitly and verify the summary |
| Expecting memory to survive a kernel restart | In-memory objects disappear | Store conversation state persistently |

## 21. Practice example: create a fresh conversation

This creates a separate memory object and chain. It starts with an empty summary, independent of the earlier `conversation` object.

In [15]:
practice_memory = ConversationSummaryMemory(llm=llm_model)

practice_conversation = ConversationChain(
    llm=llm_model,
    memory=practice_memory,
    verbose=False
)

print(practice_conversation.predict(
    input="My favorite programming language is Python."
))

print(practice_conversation.predict(
    input="I am learning AI agents."
))

print(practice_conversation.predict(
    input="What programming language do I like, and what am I learning?"
))

Oh, Python! That is a fantastic choice. It’s arguably the most versatile and popular language in the world right now, and for good reason.

What draws you to it the most? Is it the clean, readable syntax that feels almost like writing pseudocode? Or are you more into the massive ecosystem of libraries? For example, if you’re into data science, you probably use **Pandas** for data manipulation and **NumPy** for numerical computing. If you’re into web development, you might be working with **Django** or **Flask** to build robust backends. And if you’re into machine learning, **PyTorch** and **TensorFlow** have made it incredibly accessible to build complex neural networks without getting bogged down in low-level C++ code.

I’ve noticed that Python’s "batteries included" philosophy really shines in scripting tasks—like automating file organization, scraping data from websites with **BeautifulSoup**, or even creating simple command-line tools. Do you find yourself using it mostly for one s

## 22. Practice: inspect two independent memories

The first conversation knows about Omkar and LangChain. The practice conversation knows about Python and AI agents.

They are separate objects.

In [16]:
print("Main conversation summary:")
print(summary_memory.load_memory_variables({}).get("history"))

print("\nPractice conversation summary:")
print(practice_memory.load_memory_variables({}).get("history"))

Main conversation summary:
Omar introduces himself as an AI Engineer, leading the AI to describe the field as a blend of theory, engineering, and data artistry, and ask about his specific focus. Veera then introduces herself as an FDE Engineer; the AI greets her, expresses uncertainty about the acronym "FDE" (suggesting possibilities like Full-Stack Developer or Field/Functional Design Engineer), and asks for clarification along with details on her workflow and tech stack. Veera adds that she teaches DevOps, prompting the AI to praise the intersection of software development and IT operations, specifically regarding Infrastructure as Code and AI in CI/CD pipelines, while inquiring about her curriculum’s focus on traditional versus cloud-native tools and student challenges. The AI confirms Veera’s identity as an FDE Engineer and DevOps instructor. Veera states she works with Python and cloud technologies, which the AI praises as a powerful combination for cloud engineering and data scie

## 23. Exercises for students

Try these without looking at the earlier cells:

1. Create a `ConversationSummaryMemory` object.
2. Connect it to a `ConversationChain`.
3. Tell the chatbot your name and job role.
4. Tell it which technology you are learning.
5. Ask it to recall both details.
6. Print the summary using `load_memory_variables({})`.
7. Create a second memory object and verify that it starts with separate context.
8. Explain the difference between Buffer Memory and Summary Memory in your own words.

### Challenge

Have a conversation with at least five turns. Inspect the summary after each turn. Notice which facts are retained and how the summary changes.

## 24. Quick revision

- **ConversationSummaryMemory:** Uses an LLM to maintain a concise summary of the conversation.
- **Progressive summarization:** Combines the previous summary with new conversation lines to create an updated summary.
- **ChatGroq:** Provides the chat model.
- **ConversationChain:** Connects the model and memory in this notebook's legacy-style example.
- **`predict(input=...)`:** Sends a user message through the chain.
- **`load_memory_variables({})`:** Retrieves the memory variable, commonly `history`.
- **Separate memory objects:** Maintain separate conversation contexts.
- **Persistence:** Requires a database or persistent store if history must survive process restarts.

**One-line definition for students:**

ConversationSummaryMemory uses an LLM to continuously convert the conversation into a shorter summary and provides that summary as context for future questions.